NAME-SHIVALI GUPTA,
ROLL NO.-MSD25016

# Data Mining Assignment - Spark Data Lake (sales.csv)

In [1]:
!pip install pyspark -q

In [2]:
import os, shutil, filecmp

os.makedirs("data_lake/bronze", exist_ok=True)
os.makedirs("data_lake/silver/preprocessed_sales", exist_ok=True)

if not os.path.exists("sales.csv"):
    from google.colab import files
    up = files.upload()
    os.rename(list(up.keys())[0], "sales.csv")

shutil.copyfile("sales.csv", "data_lake/bronze/sales.csv")
print("bronze same as original:", filecmp.cmp("sales.csv", "data_lake/bronze/sales.csv", shallow=False))

Saving sales (3).csv to sales (3).csv
bronze same as original: True


In [3]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.appName("sales_data_lake").master("local[*]")
         .config("spark.sql.ansi.enabled", "false").getOrCreate())

raw = spark.read.csv("data_lake/bronze/sales.csv", header=True, inferSchema=False)
before = raw.count()

raw.printSchema()
raw.show(5, truncate=False)
print("total records:", before)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_percent: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+---

In [4]:
# duplicates and nulls
print("exact duplicate rows:", before - raw.distinct().count())
print("order_ids appearing more than once:", raw.dropDuplicates().groupBy("order_id").count().filter("count > 1").count())

raw.select([F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c)
            for c in raw.columns]).show()

exact duplicate rows: 10
order_ids appearing more than once: 8
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|       0|          0|           10|      0|       0|       0|         0|               0|             8|   8|    0|         0|           0|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



In [5]:
# text columns
text_cols = ["customer_name", "product", "category", "payment_method", "city", "state", "order_status"]

for c in text_cols:
    spaces = raw.filter(F.col(c) != F.trim(F.regexp_replace(F.col(c), r"\s+", " "))).count()
    variants = raw.select(c).distinct().count()
    unique_vals = raw.select(F.lower(F.trim(F.col(c)))).distinct().count()
    print(c, "| rows with extra spaces:", spaces, "| distinct:", variants, "| distinct after lower+trim:", unique_vals)

raw.groupBy("category").count().show()
raw.groupBy("city").count().show(30)

customer_name | rows with extra spaces: 3 | distinct: 306 | distinct after lower+trim: 303
product | rows with extra spaces: 0 | distinct: 57 | distinct after lower+trim: 57
category | rows with extra spaces: 0 | distinct: 13 | distinct after lower+trim: 8
payment_method | rows with extra spaces: 0 | distinct: 6 | distinct after lower+trim: 6
city | rows with extra spaces: 3 | distinct: 19 | distinct after lower+trim: 12
state | rows with extra spaces: 0 | distinct: 9 | distinct after lower+trim: 9
order_status | rows with extra spaces: 0 | distinct: 5 | distinct after lower+trim: 5
+---------------+-----+
|       category|count|
+---------------+-----+
|         SPORTS|    1|
|         beauty|    2|
|         Sports|  136|
|        Grocery|  136|
|    Electronics|  103|
|       Clothing|  108|
|          Books|  128|
|    ELECTRONICS|    1|
|Home Appliances|  115|
|      Furniture|  132|
|         sports|    1|
|         Beauty|  136|
|       CLOTHING|    1|
+---------------+-----+

+

In [6]:
# numeric columns
num = (raw.withColumn("q", F.col("quantity").cast("double"))
          .withColumn("p", F.col("unit_price").cast("double"))
          .withColumn("d", F.col("discount_percent").cast("double")))

print("quantity <= 0:", num.filter("q <= 0").count())
print("unit_price <= 0:", num.filter("p <= 0").count())
print("discount < 0 or > 100:", num.filter("d < 0 or d > 100").count())

num.filter("q <= 0 or p <= 0 or d < 0 or d > 100") \
   .select("order_id", "quantity", "unit_price", "discount_percent").show(30)

quantity <= 0: 12
unit_price <= 0: 8
discount < 0 or > 100: 8
+--------+--------+----------+----------------+
|order_id|quantity|unit_price|discount_percent|
+--------+--------+----------+----------------+
|  100034|       7|      1720|              -5|
|  100102|       7|         0|              40|
|  100114|       0|      2920|              15|
|  100140|       4|       790|              -5|
|  100143|       0|      1030|              15|
|  100170|       6|         0|               0|
|  100198|      -2|      2160|              12|
|  100217|       1|       650|              -5|
|  100235|       7|      -500|               0|
|  100284|       2|      -500|              30|
|  100310|       9|      9280|             120|
|  100363|       3|       900|             120|
|  100365|       0|       780|              15|
|  100375|       0|       460|              10|
|  100458|      -2|       800|              12|
|  100487|       4|      -500|              25|
|  100501|       0|       

In [7]:
# dates
bad_dates = raw.filter(F.to_date("order_date", "yyyy-MM-dd").isNull())
print("dates not in valid yyyy-MM-dd:", bad_dates.count())
bad_dates.select("order_id", "order_date").show()

dates not in valid yyyy-MM-dd: 6
+--------+----------+
|order_id|order_date|
+--------+----------+
|  100097|31/02/2026|
|  100358|2025/07/14|
|  100448|not-a-date|
|  100634|2025-02-30|
|  100637|2026-13-05|
|  100969|09-15-2026|
+--------+----------+



In [8]:
df = raw.dropDuplicates()
after_dedup = df.count()

for c in text_cols:
    df = df.withColumn(c, F.trim(F.regexp_replace(c, r"\s+", " ")))
    df = df.withColumn(c, F.when(F.col(c) == "", None).otherwise(F.col(c)))

for c in ["customer_name", "category", "city"]:
    df = df.withColumn(c, F.initcap(c))

df = df.fillna({"customer_name": "Unknown", "city": "Unknown", "payment_method": "Unknown"})

df = (df.withColumn("quantity", F.col("quantity").cast("int"))
        .withColumn("unit_price", F.col("unit_price").cast("double"))
        .withColumn("discount_percent", F.col("discount_percent").cast("double")))

df = df.filter("quantity > 0 and unit_price > 0")
df = df.withColumn("discount_percent",
                   F.when(F.col("discount_percent").between(0, 100), F.col("discount_percent")).otherwise(0.0))

df = df.withColumn("order_date", F.coalesce(
        F.to_date("order_date", "yyyy-MM-dd"),
        F.to_date("order_date", "yyyy/MM/dd"),
        F.to_date("order_date", "MM-dd-yyyy")))

print("after removing duplicates:", after_dedup)
print("after full preprocessing:", df.count())

after removing duplicates: 990
after full preprocessing: 970


In [9]:
df.coalesce(1).write.mode("overwrite").option("header", True).csv("data_lake/silver/preprocessed_sales")

part = [f for f in os.listdir("data_lake/silver/preprocessed_sales") if f.startswith("part-") and f.endswith(".csv")][0]
shutil.copyfile("data_lake/silver/preprocessed_sales/" + part, "data_lake/silver/sales_silver.csv")

'data_lake/silver/sales_silver.csv'

In [10]:
silver = spark.read.csv("data_lake/silver/preprocessed_sales", header=True, inferSchema=True)
silver.show(10, truncate=False)

print("rows before preprocessing:", before)
print("rows after preprocessing:", silver.count())

silver.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in silver.columns]).show()
silver.groupBy("category").count().show()

+--------+-----------+--------------+----------------------+-----------+--------+----------+----------------+----------------+---------+-------------+----------+------------+
|order_id|customer_id|customer_name |product               |category   |quantity|unit_price|discount_percent|payment_method  |city     |state        |order_date|order_status|
+--------+-----------+--------------+----------------------+-----------+--------+----------+----------------+----------------+---------+-------------+----------+------------+
|100124  |C0006      |Aditi Tripathi|Football              |Sports     |1       |1670.0    |35.0            |Cash on Delivery|Jaipur   |Rajasthan    |2026-06-29|Delivered   |
|100530  |C0140      |Sneha Sharma  |Competitive Exam Guide|Books      |7       |950.0     |5.0             |Cash on Delivery|Pune     |Maharashtra  |2026-06-09|Delivered   |
|100576  |C0265      |Pranav Joshi  |Bedside Table         |Furniture  |9       |3570.0    |40.0            |Debit Card      

**Preprocessing done**
- removed exact duplicate rows
- filled missing customer name, city and payment method with "Unknown"
- trimmed extra spaces and fixed capitalisation of customer name, category and city
- dropped rows where quantity or unit price was zero/negative, and set discounts outside 0-100 to 0
- converted dates to yyyy-MM-dd, dates that are not real dates (like 31/02/2026 or not-a-date) were set to null
- saved the final data in the silver folder